In [ ]:
import pandas as pd
import re

# ============================================================
# 1. LOAD BOTH SHEETS
# ============================================================

file_path = r"GABUNGAN_IBUKOTA.xlsx"

df_phi = pd.read_excel(
    file_path,
    sheet_name="Data PHI"
)

df_phk = pd.read_excel(
    file_path,
    sheet_name="PHK Luar PHI - Jakarta Timur"
)


# ============================================================
# 2. COUNT PARTIES
# ============================================================

def count_parties(para_pihak):

    if pd.isna(para_pihak):
        return 0

    text = str(para_pihak).strip()

    if not text:
        return 0

    total = 0

    # --------------------------------------------------------
    # Split different party groups using "|"
    # --------------------------------------------------------
    sections = text.split("|")

    for section in sections:

        section = section.strip()

        if not section:
            continue

        # ----------------------------------------------------
        # Remove the label at the beginning of each section
        #
        # Examples:
        # Penggugat:
        # Tergugat:
        # Pihak:
        # Penuntut Umum:
        # Terdakwa:
        # ----------------------------------------------------
        section = re.sub(
            r'^\s*[^:|]+:\s*',
            '',
            section
        )

        # ----------------------------------------------------
        # Split multiple parties using ";"
        # ----------------------------------------------------
        parties = section.split(";")

        for party in parties:

            party = party.strip()

            if not party:
                continue

            # ------------------------------------------------
            # Remove numbering:
            #
            # 1.NAME
            # 2.NAME
            # 3.NAME
            # ------------------------------------------------
            party = re.sub(
                r'^\s*\d+\.\s*',
                '',
                party
            ).strip()

            if party:
                total += 1

    return total


# ============================================================
# 3. APPLY TO BOTH SHEETS
# ============================================================

df_phi["Jumlah Pihak Terlibat"] = (
    df_phi["Para Pihak"]
    .apply(count_parties)
)

df_phk["Jumlah Pihak Terlibat"] = (
    df_phk["Para Pihak"]
    .apply(count_parties)
)


# ============================================================
# 4. CALCULATE TOTALS
# ============================================================

total_phi = df_phi["Jumlah Pihak Terlibat"].sum()

total_phk = df_phk["Jumlah Pihak Terlibat"].sum()

total_semua = total_phi + total_phk


# ============================================================
# 5. DISPLAY RESULTS
# ============================================================

print("Data PHI")
print(f"Jumlah perkara : {len(df_phi):,}")
print(f"Jumlah pihak   : {total_phi:,}")

print("\nPHK Luar PHI - Jakarta Timur")
print(f"Jumlah perkara : {len(df_phk):,}")
print(f"Jumlah pihak   : {total_phk:,}")

print("\nTOTAL KESELURUHAN")
print(f"Jumlah perkara : {len(df_phi) + len(df_phk):,}")
print(f"Jumlah pihak   : {total_semua:,}")